# __Méthodologie en Génétique Humaine__

## **TP: Étude d'association pan-génomique dans le diabète de type 1**

*Claire Vandiedonck*

<u>Déroulé de l'ensemble du TP :</u>
<blockquote><ul>
    <i><b>Activités préparatoires : voir <code>T1D_GWAS_prepatory_activity.ipynb</code></i></b>

0. Synopsis<br>
1. Première partie: Evaluation de la faisabilité de l'étude<br>
<ul>
    1.A. Identification des régions déjà connues pour leur association au DT1 de manière significative<br>
    1.B. Calcul de puissance pour retrouver les mêmes régions ou d'autres variants avec effets similaires dans une nouvelle cohorte<br>
</ul>
2. Deuxième partie: Formatage de fichiers de génétique et découverte du logiciel PLINK<br>

<br><span style="color:blue">
    <i><b>Séance de TP : ce notebook <code>T1D_GWAS_MEGM1-MGH.ipynb</code></i></b><br>
3. Troisième partie: Statistiques descriptives et QC<br>
4. Quatrième partie: Analyse d'association génétique cas-contrôles<br>
5. Cinquième partie: Analyse d'association familiale<br>
</span>

</ul></blockquote>

## Avant d'aller plus loin
---
<div class="alert alert-block alert-danger"><b>Attention:</b> 
Ne travaillez pas directement sur ce notebook pour ne pas le perdre. Dupliquez-le et renommez-le par exemple en ajoutant vos initiales et travaillez sur cette nouvelle copie. Pour ce faire, dans le panneau de gauche, faites un clic droit sur le fichier et sélectionnez "Duplicate". Puis, toujours dans la colonne de gauche, faites un clic droit sur cette copie et sélectionnez "rename" pour changer le nom. Ouvrez ensuite cette nouvelle version en double cliquant dessus. Vous êtes prêt(e) à démarrer! <br>
<br>
<b>N'oubliez pas de sauvegarder régulièrement votre notebook</b>: <kbd>Ctrl</kbd> + <kbd>S</kbd>. ou en cliquant sur l'icone 💾 en haut à gauche de votre notebook ou dans le Menu du JupyterLab "File puis "Save Notebook"!
</div>

<div class="alert alert-block alert-info"> 
   
<b>Rappel :</b> Vous pouvez prendre des notes directement dans ce notebook, en ajoutant une cellule Markdown en cliquant sur l'icône <kbd>➕</kbd> dans la barre des menus, et en choisissant son format dans le menu déroulant. 
Les astuces pour utiliser les cellules des notebooks sont rappelées tout en bas dans un cadre bleu ⏬.
</div>

Dans ce TP, vous lancerez chaque cellule l'une après l'autre dans l'ordre. Si vous revenez en arrière et rééxécuter une cellule, veillez à relancer les suivantes. Les cellules sont numérotées dans leur ordre d'éxécution.

__*=> A propos de ce jupyter notebook*__

Pour ce TP, nous aurons besoin d'éxécuter du code **bash** et **R**. Nous allons utiliser un notebook avec un noyau **python** et indiquer que les cellules doivent être éxécutées en bash (natif) en ajoutant `%%bash` au début de chaque cellule, et en ajoutant `%%R` au début de chaque cellule `R` après avoir installé et chargé `Rpy2` au début du notebook. Cela restera très lisible.

Avant de commencer, nous devons donc charger le module `Rpy2` avec la commande suivante.

In [ ]:
# cell 1
%load_ext rpy2.ipython

Vous pouvez éxécuter cette commande python pour connaître votre répertoire de travail.

In [ ]:
# cell 2
import os 
os.getcwd()

<div class="alert alert-block alert-info"><b>Le résultat devrait ressembler à :</b>"/srv/home/mylogin/meg_m1_ghm_gwas" avec votre login. Si ce n'est pas le cas, changez de répertoire avec la commande "os.chdir('path')" ou applez votre enseignant à l'aide!</div>

## **III.Troisième partie: Statistiques descriptives et QC** *(~ 45')*
---
    

<div class="alert alert-block alert-warning"><b>POUR RAPPEL</b><br>
Vous avez appris à identifier les commandes de base dans PLINK pour lire les fichiers d'inputs, ajouter un suffixe aux fichiers d'outputs et convertir les fichiers entre les formats plats et binaires. Le rappel de ces commandes, tiré de la liste des commandes PLINK disponibles à <a href="https://www.cog-genomics.org/plink/1.9/data">ce lien</a>, est résumé ci-dessous:<br> 

| Option                        | Parameter/default       | Description                      |
|-------------------------------|-------------------------|----------------------------------|
| Basic input/output            |                         |                                  |
| --file                        |  {plink}                |   Specify .ped and .map files    |
| --ped                         |   {plink.ped}           |   Specify .ped file              |
| --map                         |   {plink.map}           |   Specify .map file              |
| --bfile                       |  {plink}                |   Specify .bed, .bim and .fam    |
| --bed                         |  {plink.bed}            |   Specify .bed file              |
| --bim                         |  {plink.bim}            |   Specify .bim file              |
| --fam                         |  {plink.fam}            |   Specify .fam file              |
| --out                         |  {plink}                |   Specify output root filename   |
| Other data management options |                         |                                  |
| --make-bed                    |                         |   Make .bed, .fam and .bim       |
| --recode                      |                         |   Output new .ped and .map files |
</div>

### **III.A. Lecture des fichiers des données de cas et contrôles**

Les fichiers des cas et de contrôles sont dans le répertoire `input` de votre environnement. Ils sont également sur moodle. Les fichiers ont les préfixes `ic_controls` pour les contrôles et `ic_cases` pour les patients diabétiques.

Chargez chaque série indépendamment dans PLINK. Avec la version PLINK 1.9, il n'est pas possible de lire les fichiers d'inputs sans executer une commande sur ces fichiers. Utilisons en une toute simple qui écrit la liste des SNPs `--write-snplist` sans modifier nos fichiers d'inputs. Je vous suggère également de mettre `output/read_controls` comme préfixe des fichiers de sortie. Faites de même avec les cas avec `output/read_cases`. Pour chaque jeu de données, lisez le fichier log ou l'équivalent qui s'affiche sous votre cellule excéutée pour répondre aux questions ci-dessous. Ne faites pas attention aux chiffres affichés dans la ligne commençant par "Calculating allele frequencies" si vous lisez le log directement sur le notebook. Cette même ligne a une apparence normale sans ses chiffres si vous lisez le fichier d'output ".log".

<span style="color:blue"><b>Q3.1.</b><i> Concernant les <b>contrôles</b> :<i></span>

In [ ]:
%%bash
#cell 3
# entrez ci-après votre commande pour les contrôles.


<span style="color:blue"><i>- Quel est le format des fichiers d'entrée ?</i></span>

<span style="color:blue"><i>- Donnez le nombre de sujets ?</i></span>

<span style="color:blue"><i>- Le statut vis à vis de la maladie est-il renseigné pour tous ?</i></span>

<span style="color:blue"><i>- Indiquez les effectifs par sexe</i></span>

<span style="color:blue"><i>- Combien de marqueurs sont inclus ?</i></span>

<span style="color:blue"><i>- Quel est le taux de génotypage ?</i></span>

<span style="color:blue"><b>Q3.2.</b><i> Concernant les <b>patients</b> :</i></span>

      

In [ ]:
%%bash
#cell 4
# entrez ci-après votre commande pour les cas; deux warings seront affichés qui ne vous gènent pas pour répondre aux questions ci-dessous.


<span style="color:blue"><i>Quel est le format des fichiers d'entrée ?</i></span>

<span style="color:blue"><i>- Donnez le nombre de sujets ?</i></span>

<span style="color:blue"><i>- Le statut vis à vis de la maladie est-il bien « cas » pour tous ?</i></span>

<span style="color:blue"><i>- Indiquez les effectifs par sexe.</i></span>

<span style="color:blue"><i>- Combien de marqueurs sont inclus ?</i></span>


<span style="color:blue"><i>- Quel est le taux de génotypage ?</i></span>

### **III.B. Fusion des fichiers des données de cas et contrôles en un seul jeu de données**

En fait, les cas et les contrôles ont été collectés par deux laboratoires différents qui les ont génotypés avec la même puce « immunochip » d'Illumina mais indépendamment sur des plates-formes différentes. Chaque laboratoire a ensuite effectué des pré-traitements des fichiers (récupération des intensités, nettoyage des SNPs, vérification des contrôles positifs et négatifs…). Ils ne sont donc pas homogènes. Le fichier des patients est relativement peu nettoyé, celui des contrôles l'est davantage.

Dans cette section, vous allez effectuer quelques manipulations de ces fichiers afin de pouvoir les combiner en un seul jeu de données comparables portant sur les mêmes marqueurs. Vous trouverez les commandes à utiliser dans la section Data management (https://www.cog-genomics.org/plink/1.9/data) du manuel de PLINK.

<div class="alert alert-block alert-danger"><b>Attention: à chaque étape de nettoyage, donnez un nouveau nom à vos fichiers!</b> 
<br>=> Si vous voulez générer des fichiers .ped et .map, la commande est : <b>--recode</b>
<br>=> Si vous voulez générer des fichiers .bed, .bim et .fam, la commande est : <b>--make-bed</b>
</div>

<span style="color:blue"><b>Q3.3.</b><i> Lorsque vous avez chargé les <b>fichiers de patients</b> dans la section précédente (III.A), qu'avez-vous remarqué pour <b>31 024 marqueurs</b> que vous n'observiez pas avec les fichiers de contrôles (message de waring de la question 3.2)? Ouvrez avec un éditeur de texte le <b>fichier « .hh »</b> généré par PLINK à cette occasion. A quoi correspond-il ? Pour vous aider, les marqueurs sont rangés en fonction de leur position chromosomique. Si vous vérifier la position du premier et du dernier marqueur(avec le fichier map), vous devriez comprendre de quels marqueurs il s'agit et comprendre le génotype observé à ces marqueurs. Pourquoi n'observiez vous pas ce problème avec les contrôles ?</b> :</i></span>

- **Retrait des SNPs de position chromosomique incertaine et des SNPs sur les gonosomes**

Dans une première étape de nettoyage, vous éliminerez de vos fichiers de patients tous les génotypes des marqueurs de ces gonosomes et tous les SNPs de position chromosomique incertaine (chromosome numéroté 0 dans `.map`). Il n'existe pas une commande unique permettant de faire cette opération automatiquement. Le plus simple est d'écrire tous les SNPs à exclure dans un fichier texte `.txt` *(déjà dans input:  "toexclude.txt")* et d'utiliser la commande `--exclude`. Alternativement, vous pouvez écrire un jeu de données par chromosome avec la commande `--chr` puis les combiner avec la commande `--merge-list` mais c'est plus long!

<span style="color:blue"><b>Q3.4.</b><i>Quelle commande utilisez-vous ? Combien de marqueurs reste-il ? Quel est le nouveau taux de génotypage moyen ?</b></i></span>

In [ ]:
%%bash
# cell 5,  entrez ci-après votre commande


- **Tentative de fusion des données de patients et contrôles**

Tentez à présent de combiner les fichiers de patients et de contrôles avec la commande : `--merge`
Nommez votre nouveau jeu de données `t1dcc` pour "t1d case-control".

In [ ]:
%%bash
# cell 6, entrez ci-après votre commande


<span style="color:blue"><b>Q3.5.</b><i>Quelle commande utilisez-vous ? Quels sont les <b>deux problèmes détectés par PLINK</b> : « warning » et « error »? Quelle est l'explication proposée par PLINK pour le second problème ?</b></i></span>

- **Exlusion des SNPs avec plusieurs positions génomiques**

Excluez de chaque jeu de données le(s) marqueur(s) avec plusieurs positions génomiques avec la commande `--exclude`.

In [ ]:
%%bash
# cell 7, entrez ci-après vos commandes


<span style="color:blue"><b>Q3.6.</b><i>Combien de marqueurs avez-vous retirés ?</b></i></span>

- **Homogénisation du brin considéré**

Pour régler le second problème, changez le brin sur le jeu de données de contrôles en utilisant la commande : `--flip` sur le fichier `merge.missnp` (ce fichier avec l'extension « .missnp » avait été automatiquement généré par PLINK lorsque vous avez essayé de merger les données).

In [ ]:
%%bash
# cell 8, entrez ci-après votre commande


- **Seconde tentative de fusion des données de patients et contrôles**

Combinez de nouveau au jeu de données de patients que vous nommerez encore t1dcc :

In [ ]:
%%bash
# cell 9, entrez ci-après votre commande


<div class="alert alert-block alert-success"><b>=> Bravo !</b><br>Vous commencez à bien comprendre les commandes PLINK. A partir de maintenant, toutes les commandes dans ce notebook vous sont données.</div>

Ici nous avons encore un souci pour 1436 (3 indiqués + 1433 autres dans le fichier log) positions génomiques pour lesquelles deux SNPs sont mappés. Leur liste est donnée dans le figier `t1dcc.log`. Pour chaque position, il faut donc ne garder qu'un seul des deux SNPs.
Le choix a été de privilégier le "rsID" lorsque le deuxième SNP n'a pas un indetifiant de type rs, ou la première occurence de rsID quand les deux SNPs ont un rs.

Comme la procédure est assez longue, je vous donne directement le résultat pour gagner du temps.

 Les fichiers ont les préfixes `t1dcc` (pour T1D Case Control). Les données fusionnées propres sont accessibles sur moodle: https://moodle.u-paris.fr/mod/folder/view.php?id=174031. Les fichiers sont aussi dans `/srv/data/pir-g2/input/`.
 Nous allons le copier dans votre répertoire `input`.

In [ ]:
%%bash
# cell 10
cp /srv/data/pir-g2/input/t1dcc* ./input

In [ ]:
%%bash
# cell 11
plink --bfile input/t1dcc --write-snplist --out output/read_t1dcc

<span style="color:blue"><b>Q3.7.</b><i> Description des fichiers et données:</i></span>

<span style="color:blue"><i>- Donnez le nombre de sujets? </i></span>
 

<span style="color:blue"><i>- Comment se répartissent-ils entre patients et contrôles? </i></span>

<span style="color:blue"><i>- Indiquer les effectifs par sexe.</i></span>

<span style="color:blue"><i>- Combien de marqueurs sont inclus ? </i></span>

<span style="color:blue"><i>- Quel est le taux moyen de génotypage par SNP? </i></span>

### **III.B. Contrôle de qualité**

Notre jeu de données ainsi prêt, vous pouvez procéder aux analyses de contrôle de qualité avec plusieurs étapes de filtres pour élminier les données de mauvaise qualité.<br>
Il est recommandé de commencer par les contrôles de qualité sur les SNPs, avant de procéder aux contrôles de qualité sur les sujets.

<div class="alert alert-block alert-danger"><b>Attention: à chaque étape de nettoyage, nous donnons un nouveau nom aux fichiers!</b> 
<br>=> Si vous voulez générer des fichiers .ped et .map, la commande est : <b>--recode</b>
<br>=> Si vous voulez générer des fichiers .bed, .bim et .fam, la commande est : <b>--make-bed</b>
</div>

##### **-> QC sur les SNPs**

- Exclusion des SNPs avec un **taux de génotypage <95 %** dans l'ensemble de l'étude

In [ ]:
%%bash
# cell 12
plink --bfile input/t1dcc --geno 0.05 --make-bed --out output/filter1

- Exclusion des SNPs avec des **taux de génotypage trop différents entre patients et contrôles**

In [ ]:
%%bash
# cell 13
plink --bfile output/filter1 --test-missing --pfilter 0.00001 --out output/missingcc

Cette 1ère commande a généré la liste des SNPs problématiques dans un fichier « missingcc.missing ». Excluez-les à présent avec cette 2nde commande  :

In [ ]:
%%bash
# cell 14
plink --bfile output/filter1 --exclude output/missingcc.missing --make-bed --out output/filter2

- Exclusion des **SNPs qui ne sont pas en équilibre de Hardy-Weinberg** chez les contrôles au seuil 10<sup>-4</sup>

In [ ]:
%%bash
# cell 15
plink --bfile output/filter2 --hwe 0.0001 --make-bed --out output/filter3

- Exclusion des SNPs rares avec une **MAF < 0.01**

In [ ]:
%%bash
# cell 16
plink --bfile output/filter3 --maf 0.01 --make-bed --out output/filter4

D'autres contrôles peuvent être effectués : vérifier que les SNPs ne sont pas dupliqués (PLINK le signale automatiquement), que les SNPs n'ont qu'une seule position génomique sur le génome, etc...

<span style="color:blue"><b>Q3.8.</b><i>Combien de marqueurs sont exclus à chaque étape? Combien reste-t-il de marqueurs après ce QC sur les SNPs ?</i></span>

<span style="color:blue"><b>Q3.9.</b><i> Pourquoi ne considère-t-on pas les SNPs rares dans cette étude ?</i></span>

##### **->QC sur les sujets**

- Exclusion des sujets avec un **taux de génotypage <95 %**

In [ ]:
%%bash
# cell 17
plink --bfile output/filter4 --mind 0.05 --make-bed --out output/filter5

- Exclusion des **sujets apparentés**

La détection des sujets apparentés peut se faire avec PLINK mais nécessite de nombreuses commandes (`--genome`, ) et est très longue à tourner.
<br>Il est plus aisé d'utiliser un logiciel dédié à cette analyse : **KING** (Kinship-based Infererence for GWAS): http://people.virginia.edu/~wc9c/KING/ . Ce logiciel est très rapide, permet de lire des fichiers binaires de PLINK. Il permet aussi d'analyser la structure de la population.
> Si vous utiliez KING en dehors d'adénine, téléchargez-le, dézippez le et copier une cmd.exe dans le même répertoire si vous travaillez sous windows ou copiez le king.exe dans votre répertoire de travail. Comme pour PLINK, les fichiers d'entrée doivent être dans le même répertoire.

Exécutez la commande suivante (<mark>elle prend quelques secondes</mark>). Comme **KING** ne génère pas de fichier `.log`, nous n'avons pas à nous inquiéter d'écraser le log précédent.

In [ ]:
%%bash
# cell 18
king -b output/filter5.bed --kinship --related --prefix output/filter5

Cette commade vous a gènèré un fichier `filter5.kin` indiquant les relations dans les familles, et un fichier `filter5.kin0` indiquant les relations entre les familles. Dans ces fichiers, la colonne « Kinship » correspond au coefficient de corrélation estimé. S'il est > 0.17, ils est probable qu'ils soient apparentés au moins au 1er degré.

La commande suivante vous permet de récupérer la liste des sujets non apparentés (<mark>elle prend quelques secondes, soyez patients!</mark>). Attention elle s'effectue toujours sur l'input de prefixe `filter5` car la commande précédente de **KING** n'a pas généré de nouveaux fichiers de pedigree. 

In [ ]:
%%bash
# cell 19
king -b output/filter5.bed --kinship --unrelated --prefix output/filter5

La commande suivante de **PLINK** vous permet de sélectionner les sujets non apparentés :

In [ ]:
%%bash
# cell 20
plink --bfile output/filter5 --keep output/filter5unrelated.txt --make-bed --out output/filter6

De nombreux autres vérifications peuvent être effectuées :
<br>On peut par exemple vérifier que les génotypes sur les chromosomes X et Y, s'ils sont disponibles, sont en accord avec le sexe des sujets. On peut également vérifier que les sujets ne sont pas en excès d'hétérozygotie, ce qui pourrait indiquer une contamination par un autre ADN. Si certains sujets ont été génotypés plusieurs fois (contrôle positif présent sur chaque plaque de génotypage), il est bon de vérifier que les génotypes sont identiques. De même, on peut avoir placé des sujets de génotype déjà connus pour tous les SNPs ou certains SNPs de la puce et vérifier la concordance.

<span style="color:blue"><b>Q3.10.</b><i> Combien de sujets sont exclus à chaque étape? Combien reste-t-il de cas et de contrôles à l'issue de ce QC ?</i></span>

<span style="color:blue"><b>Q3.11.</b><i> Quel contrôle essentiel pour une étude d'association cas-contrôle n'a-t-on pas encore fait ?</i></span>

Nous le ferons plus tard, après une première tentative de GWAS.

## **IV. Quatrième partie: Analyse d'association génétique cas-contrôles** *(~ 45')* 
---

### **IV.A. 1er essai de test d'association allélique pangénomique**

**-> test de Chi2**

Effectuez maintenant le test d'association allélique pangénomique avec les arguments:<br>
```--assoc```: un test de Chi2<br>
```--adjust```: génère les pvalues ajustées également<br>
```--ci 0.95```: afin d'avoir l'IC à 95% de l'OR

In [ ]:
%%bash
# cell 21
plink --bfile output/filter6 --assoc --ci 0.95 --adjust --out output/allelictest

<span style="color:blue"><b>Q4.1.</b><i> Quels sont les fichiers de sortie produits par cette commande ? A quoi correspondent-ils ?</i></span>

<div class="alert alert-block alert-warning"><b>Au lieu de faire un test de Chi2 qui repose sur une approximation des distributions des allèles selon le statut, il est préférable de réaliser un test exact de Fisher qui prend en compte la distribution réelle de vos données.</b> Ce test peut-être très long à réaliser. Mais avec les outils informatiques, c'est désormais plus accessible. Nous allons donc refaire cette même étude d'association avec le test exact de Fisher. Au lieu de taper l'option <code>--assoc</code> il faut taper l'option <code>--fisher</code>. Les deux fichiers de résultats obtenus contiendront par défaut le mot "fisher". Il serait donc répétitif de le mettre dans le préfixe de vos fichiers de sortie que je vous propose de simplifier par "allelic".</div>

**-> test exact de Fisher**

In [ ]:
%%bash
# cell 22
plink --bfile output/filter6 --fisher --ci 0.95 --adjust --out output/allelic

-> Vous voyez, ca a été rapide avec PLINK1.9 et adenine!

Par défaut PLINK fournit des outputs avec des espaces multiples comme séparateur de colonnes, afin de les aligner visuellement. Mais pour certaines autres applications, il peut être utile de le convertir en format avec des tabulations comme séparateur de colonne. Dans la documentation de PLINK 1.9 à ce lien https://www.cog-genomics.org/plink/1.9/other, ils expliquent comment y parvenir avec trois commandes que nous enchaînons avec des pipes:
- une première commande avec `sed` pour remplacer l'espace en début de chaque ligne par rien
- une seconde commande avec `sed` pour remplacer l'espace en fin de chaque ligne par rien
- une dernière commande avec `tr` pour remplacer (translate) les séries de 1 à plusieurs espaces en une seule tabulation
Pour pouvoir ouvrir ce fichier directement avec un outil de tabulation (excel, librecalc ou TSVTable sur le jupyterLab), nous sauvegardons ce nouveau fichier en ajoutant l'extension `.tsv`.

Nous appliquons ces commandes ci-dessous pour les deux fichiers de sortie principaux de plink:
- allelic.assoc.fisher trié par position physique
- allelic.assoc.fisher.adjusted trié par pvalue croissante

In [ ]:
%%bash
# cell 23
cat output/allelic.assoc.fisher | sed 's/^[[:space:]]*//g' | sed 's/[[:space:]]*$//g' | tr -s ' ' '\t' > output/allelic.assoc.fisher.tsv
cat output/allelic.assoc.fisher.adjusted | sed 's/^[[:space:]]*//g' | sed 's/[[:space:]]*$//g' | tr -s ' ' '\t' > output/allelic.assoc.fisher.adjusted.tsv

**-> visualisation des résultats et QC**

A présent, nous allons explorer visuellement les résultats du test exact de Fisher en utilisant le paquet **qqman** de **R** (https://cran.r-project.org/package=qqman).

Comme lorsque l'on lançait des commandes bash, il est nécessaire dans ce notebook avec un noyau python, de spécifier que la cellule sera exéutée dans un autre langage R que python. Pour ce faire, <mark>après avoir bien lancé la 1ère cellule de ce notebook qui active le module `Rpy2`</mark>, il suffit alors d'écrire `%%R` sur la 1ère ligne de la cellule. Dans un notebook avec un noyau R, ce ne serait pas nécessaire.

Chargeons donc le paquet R depuis votre librairie et vérifions. En principe, vous avez la version `0.1.4` de qqman qui a déjà été installée dans votre environnement JupyterLab.

In [ ]:
%%R
# cell 24
library(qqman)
sessionInfo()

Pour savoir comment utiliser qqman, vous pouvez consulter la vignette de ce paquet au lien suivant: https://cran.r-project.org/web/packages/qqman/vignettes/qqman.html

- Nous allons commencer par charger dans R les résultats de l'étude d'association allélique obtenue avec PLINK avec la fonction `read.table()`.

In [ ]:
%%R
# cell 25
gwasResults <- read.table("output/allelic.assoc.fisher",header=T)# on peut charger directement dans R la version avec les espaces comme séparateur de champs!
str(gwasResults)# pour afficher la strture de l'objet
head(gwasResults)# pour afficher les 6 premières lignes

- Tracez ensuite votre Manhattan plot avec la fonction `manhattan()` avec les commandes suivantes:

In [ ]:
%%R
# cell 26
manhattan(gwasResults)# une première fois pour qu'il s'affiche dans le notebook
png("output/Fisher_test_manhattan.png")# on recommence pour sauvegarder la figure à part
manhattan(gwasResults)
dev.off()

- Il est également recommandé de tracer un qqplot, qui visualise la conformité de la distribution des p-values à celle attendue selon une loi uniforme en cas d'absence d'association (hypothèse nulle). Le même paquet **qqman** vous permet de tracer ce qqplot avec la fonction `qq()`.

In [ ]:
%%R
# cell 27
qq(gwasResults$P, main="Q-Qplot of P-values")# une première fois pour qu'il s'affiche dans le notebook

png("output/Fisher_test_qqplot.png")# on recommence pour sauvegarder la figure à part
qq(gwasResults$P, main="Q-Qplot of exact Fisher test P-values")
dev.off()

- On réajuste les axes pour avoir la ligne rouge en diagonale

In [ ]:
%%R
# cell 28
qq(gwasResults$P, main="Q-Qplot of P-values",xlim=c(0,max(-log10(gwasResults$P))))

<span style="color:blue"><b>Q4.2. </b><i>Que pensez-vous de ces résultats à l'échelle pangénomique ?</i></span>

<span style="color:blue"><b>Q4.3. </b><i>Dans le fichier « allelic.log » généré par PLINK, quelle valeur d'inflation factor obtenez-vous ? Une valeur proche de 1 indique une population homogène. Qu'en concluez-vous sur votre échantillon et les résultats obtenus ?</i></span>

### **IV.B. GWAS en prenant compte de la stratification de population**

#### **1. Evaluation de la stratification de population**

Pour évaluer la stratification de population, vous réaliserez une ***MDS*** = multi-dimensional-scaling analysis avec le logiciel **KING** (https://people.virginia.edu/~wc9c/KING/kingpopulation.html). Ce type d'analyse est similaire à une ACP, mais au lieu d'utiliser les covariances entre les variables, la MDS prend en compte la distance Euclidienne et est plus adaptée pour rapprocher les individus.

- Lancez la commande suivante pour faire une **analyse MDS** (cette commande peut prendre quelques secondes):

In [ ]:
%%bash
# cell 29
king -b output/filter6.bed --mds --pcs 20 --prefix output/stratif

-> Regardez le fichier `stratifpc.txt`. Les colonnes 7 à 26 correspondent aux 20 premiers vecteurs de cette analyse MDS (10 par défaut que nous avons modifié à 20 avec l'option `--pcs`).

- A présent, dans R, nous allons **visualiser cette analyse MDS** en séparant artificiellement la représentation des contrôles (les 357 premiers dans le fichier) et les contrôles (de 358 à 572), pour mieux visualiser les différences.

In [ ]:
%%R
# cell 30
mds <- read.table("output/stratifpc.txt", header=T, stringsAsFactors=FALSE)
#png("output/mds.png")
par(mfrow=c(2,2))
plot(mds$PC1, mds$PC2, type="n", xlab="PC1", ylab="PC2", main="MDS PC1-PC2 in Controls")
points(mds$PC1[1:357], mds$PC2[1:357], col="black")
plot(mds$PC1, mds$PC2, type="n",xlab="PC1", ylab="PC2", main="MDS PC1-PC2 2 in Cases" )
points(mds$PC1[358:572], mds$PC2[358:572], col="red")
plot(mds$PC2, mds$PC3, type="n", xlab="PC2", ylab="PC3", main="MDS PC2-PC3 in Controls")
points(mds$PC2[1:357], mds$PC3[1:357], col="black")
plot(mds$PC2, mds$PC3, type="n", xlab="PC2", ylab="PC3", main="MDS PC2-PC3 3 in Cases")
points(mds$PC2[358:572], mds$PC3[358:572], col="red")
#dev.off

<span style="color:blue"><b>Q4.4. </b><i>Que pensez-vous de ces graphiques ?</i></span>

#### **2. GWAS corrigé pour la stratification de population**

Nous allons utiliser les vecteurs propres de l'analyse MDS comme covariable dans l'étude d'association en utilisant un modèle logistique dans PLINK sous un modèle additif.

La prise en compte des 20 composantes de la MDS peut prendre jusqu'à 1h30 sur des ordinateurs classiques. Sur adenine, elle devrait prendre <mark>2 à 5 minutes max</mark>. Le fichier `stratif.txt` est lu comme un fichier "phenotype" de PLINK avec chaque colonne 3 à n lue comme une covariable. Les vecteurs de l'analyses MDS correspondent seraient ici les covariables 5 à 24 pour PLINK avec l'argument `--covar-number`. Il est plus simple de donner leur nom avec l'argument `--covar-name` dans la commande ci-dessous.
> Si toutefois vous vouliez tester le TP sur une autre machine qu'adenine, nous vous recommandons de ne prendre en compte que les 2 ou 5 premières composantes, ce qui prend 5 à 15 minutes seulement. Changez juste le numéro des covariables dans la commande.

In [ ]:
%%bash
# cell 31
plink --bfile output/filter6 --logistic --adjust --ci 0.95 --covar output/stratifpc.txt --covar-name PC1-PC20 --hide-covar --out output/aftermds_PC20

- De nouveau, nous **sauvegardons une version `.tsv` des deux fichiers de sortie**:  `.logistic` trié par position physique, et `.logistic.adjusted` trié par pvalue.

In [ ]:
%%bash
# cell 32
cat output/aftermds_PC20.assoc.logistic | sed 's/^[[:space:]]*//g' | sed 's/[[:space:]]*$//g' | tr -s ' ' '\t' > output/aftermds_PC20.assoc.logistic.tsv
cat output/aftermds_PC20.assoc.logistic.adjusted | sed 's/^[[:space:]]*//g' | sed 's/[[:space:]]*$//g' | tr -s ' ' '\t' > output/aftermds_PC20.assoc.logistic.adjusted.tsv

- De nouveau, **visualisez vos résultats** avec un Manhattan plot et un qq-plot dans R:

In [ ]:
%%R
# cell 33
library(qqman)
gwasResultsAfterMDS <- read.table("output/aftermds_PC20.assoc.logistic", header=T)
#png("output/gwas_after_stratif.png")
par(mfrow=c(2,1))
manhattan(gwasResultsAfterMDS[! is.na(gwasResultsAfterMDS$P),])
qq(gwasResultsAfterMDS$P, main="Q-Qplot of P-values")
#dev.off()

<span style="color:blue"><b>Q4.5. </b><i>La prise en compte de la stratification vous semble-t-elle efficace ? Complètement efficace?</i></span>

<span style="color:blue"><b>Q4.6. </b><i>Comparez les régions associées à celles déjà connues en vous référant aux articles (Onengut-Gumuscu S, et al., 2015, Robertson CC et al., 2021; présents dans l'environnement).</i></span>

Une des difficulté de la puce immunochip est qu'elle ne contient pas de SNPs de référence "neutres" pour les études de population. Elle est au contraire enrichie en SNPs dans des régions du génome impliquées dans la réponse immunitaire, donc soumis à sélection.


<span style="color:blue"><b>Q4.7. </b><i>Quel type d'étude d'association permettrait de surmonter cette difficulté?</i></span>

> ***Pour aller plus loin:***
>Si vous les souhaitez, vous pouvez réaliser le même test avec un autre modèle que le modèle additif en ajoutant les arguments: 	`--dominant / --recessive / -- genotypic`<br>
>Vous pouvez utiliser un autre logiciel, GCTA  pour Genome-wide Complex Trait Analysis  (http://cnsgenomics.com/software/gcta/index.html) de plus en plus populaire en génétique multifactorielle, en particulier pour estimer la variance phénotypique expliquée par les différentes régions associées.

## **V - Cinquième partie: Étude d'association familiale** *(~ 20-30')*

---
Vous allez étendre l'étude d'association de type cas-contrôles réalisée précédemment à une cohorte familiale de patients atteints de diabète de type 1. En fait, pour la plupart des patients étudiés dans l'étude d'association cas-contrôles ci-dessus, leurs parents sains ont également été collectés et génotypés avec la puce immunochip.

Les données concernant les autosomes ont déjà été formatées pour être utilisées dans PLINK.
Plusieurs contrôles qualité ont été effectués en amont:

- erreurs mendéliennes identifiées: retrait des familles ou individus problématiques si plusieurs marqueurs sont concernés
- SNPs en déséquilibre de Hardy-Weinberg éliminés
- SNPs avec une MAF < 0.01 éliminés
- SNPs et individus avec taux de génotypage insuffisant (<95%) éliminés 

Les données ainsi nettoyées pour ces familles nucléaires (ou trios) sont disponibles sur moodle avec le préfixe `ic_trio`. 

- Faites tourner l'étude d'**association familiale de type TDT** avec l'argument `--tdt` dans la commande ci-dessous:

In [ ]:
%%bash
# cell 34
plink --bfile input/ic_trio --tdt --adjust --ci 0.95 --out output/tdttrio

<span style="color:blue"><b>Q5.1. </b><i>Identifier les informations descriptives relatives aux SNPs et aux sujets inclus?</i></span>

- Ouvrez le fichier de sortie `tdttrio.tdt` qui recence les résultats ordonnés par position génomique. Dans cette analyse, considérez la colonne P pour connaitre le niveau de signification statistique non corrigé. Ouvrez aussi le fichier `tdttrio.tdt.adjusted` qui contient les résultats triés en fonction du niveau de signification croissant.

Nous en sauvegardons également une version `.tsv` au cas où.

In [ ]:
%%bash
# cell 35
cat output/tdttrio.tdt | sed 's/^[[:space:]]*//g' | sed 's/[[:space:]]*$//g' | tr -s ' ' '\t' > output/tdttrio.tdt.tsv
cat output/tdttrio.tdt.adjusted | sed 's/^[[:space:]]*//g' | sed 's/[[:space:]]*$//g' | tr -s ' ' '\t' > output/tdttrio.tdt.adjusted.tsv

- Utiliser qqman sous R pour tracer le **qqplot et le manhattan plot** que vous pourrez sauvegarder.

In [ ]:
%%R
# cell 36
tdt <- read.table("output/tdttrio.tdt", header=T)
head(tdt)

In [ ]:
%%R
# cell 37
#png("output/manhattan_trios.png")
manhattan(tdt[! is.na(tdt$P),], p="P")
#dev.off()
#png("output/qqplot_trios.png")
qq(tdt$P)
#dev.off()

<span style="color:blue"><b>Q5.2. </b><i>Que pensez-vous de ces résultats ?</i></span> 

<span style="color:blue"><b>Q5.3. </b><i>En dehors du CMH (chr6 entre 27 et 33 Mb), combien de SNPs ou de régions sont associés de manière significative (5x10<sup>-8</sup>)? Combien de SNPs ou de régions sont associés de manière suggestive (10<sup>-5</sup>)?</i></span> 

<span style="color:blue"><b>Q5.4. </b><i>Les SNPs associés à la maladie et les régions correspondantes sont-ils les mêmes que ceux de l'étude cas-contrôles?</i></span>

<span style="color:blue"><b>Q5.5. </b><i>Les SNPs associés à la maladie et les régions correspondantes sont-ils les mêmes que ceux déjà connus dans la littérature? </i></span>

Vous pourriez aussi explorer si les SNPs associés à la maladie, et les régions correspondantes, ont déjà été associés à d'autres maladies ou traits. Pour les autres maladies ou traits, vous pouvez consulter le GWAS catalogue https://www.ebi.ac.uk/gwas/, ClinVar https://www.ncbi.nlm.nih.gov/clinvar/, ou  les bases de données de variants comme dbSNP (https://www.ncbi.nlm.nih.gov/snp/ ou de gènes comme Gene du ncbi https://www.ncbi.nlm.nih.gov/gene/ ou Genecards https://www.genecards.org/ voire d'eQTLs comme GTEx https://gtexportal.org/home/. Vous pouvez aussi voir s'ils sont en DL avec d'autres SNPs qui eux avec la suite LDlink https://ldlink.nih.gov/.

<div class="alert alert-block alert-success"><b>=> Bravo!</b><br>

Vous avez réussi à mener votre 1er GWAS de bout en bout!</div>

<div class="alert alert-block alert-warning"><b>Pour aller plus loin: </b>
    
- Le logiciel <b>LocusZoom</b> (http://locuszoom.sph.umich.edu/locuszoom/genform.php?type=yourdata) vous permet de réaliser un zoom du manhattan plot centré sur un SNP ou un gène donné en ligne en renseignant le fichier de sortie de PLINK. La figure est générée dans un format .pdf.
Pour ce faire:<br>
i. télécharger le fichier de résultats de l'étude d'association trié par position physique<br>
ii. dans la 1ère boîte de locuzoom, cliquez à droite sur "PLINK data" en bleu -> cela spécificiera le bon format<br>
iii. toujours dnas la 1ère boîte, dans "Path to Your File": choisissez votre fichier de résultat<br>
iv. dans la seconde boîte, sélectionnez soit le SNP d'intérêt, soit un gène soit une région<br>
v. dans les options, sélectionnez un panel pertinent pour afficher le DL de la région.<br>

<mark><b>Attention:</b> notre jeu de données est sur la version <b>hg19/GRCHh37</b> du génome</mark>

Vous pouvez utiliser cet outil en ligne pour la ou les région(s) associée(s).
Vous pouvez aussi estimer le déséquilibre de liaison local avec PLINK et consulter le DL rapporté dans les populations de référence, par exemple en consultant par exemple les outils de la suite DLink https://ldlink.nih.gov/.    
    
    
- D'autres logiciels performants vous permettent d'éffectuer des <b>analyses d'association familiales</b>: UNPHASED (https://sites.google.com/site/fdudbridge/software/unphased-3-1) et BEAGLE (http://faculty.washington.edu/browning/beagle/beagle.html).<br><br>
    
- Vous pouvez également faire des analyses combinant plusieurs SNPs ou conditionnées sur certains SNPs.
</div>

***Références***<br>
1. Trynka G, et al. Dense genotyping identifies and localizes multiple common and rare variant association signals in celiac disease. Nature Genetics 43, 1193–1201 (2011) doi.org/10.1038/ng.998<br>
2. Parkes M, et al. Genetic insight into common pathways and complex relationships among immune-mediated diseases. Nature Genetics 14, 661-673 (2013) doi.org/10.1038/nrg3502<br>
3. Marees AT et al. A tutorial on conducting genome‐wide association studies: Quality control and statistical analysis. Int J Methods Psychiatr Res. 27:e1608. (2018) doi.org/10.1002/mpr.1608<br>
4. Onengut-Gumuscu S et al. Fine mapping of type 1 diabetes susceptibility loci and evidence for colocalization of causal variants with lymphoid gene enhancers. Nature Genetics 47, 381–386 (2015) doi.org/10.1038/ng.3245<br>
5. Robertson CC et al. (2021). Fine-mapping, trans-ancestral and genomic analyses identify causal variants, cells, genes and drug targets for type 1 diabetes. Nature Genetics 53, 962–971 (2021). doi.org/10.1038/s41588-021-00880-5

---
---

<div class="alert alert-block alert-info"> 
    
<b>Rappel</b>, dans un notebook :

- la combinaison de touches <kbd>Ctrl</kbd>+<kbd>Entrée</kbd> exécute une cellule.<br>
- la combinaison de touches <kbd>Shift</kbd>+<kbd>Entrée</kbd> exécute une cellule puis passe à la suivante. C'est équivalent à cliquer sur l'icone ▶️ dans la barre de menu du notebook.<br>
- la combinaison de touches <kbd>Alt</kbd>+<kbd>Entrée</kbd> exécute une cellule puis en crée une nouvelle (vide) en dessous.<br>

Pour ajouter une cellule, vous pouvez aussi cliquer sur l'icone ➕ dans la barre de menu du notebook.
- vous pouvez déplacer les cellules en les glissant pour les réorganiser les unes en-dessous des autres.<br>
- vous pouvez ajouter des commentaires, soit en commençant la ligne par un "#" dans une cellule de code (ces lignes ne seront pas éxécutées), soit dans une nouvelle céllule de type Markdown.<br>
- vous sélectionner le type de cellule dans le menu en haut de votre Vnotebbok.<br>
    - "Code" pour entrer des lignes de commande à éxécuter <br>
    - "Markdown pour ajouter du texte balisé qui peut être formatté<br>
- pour modifier une cellule ̀Markdown, double-cliquez dessus<br>
- attention à la casse des caractères pour les cellules de code<br>
- évitez les caractères spéciaux dans les cellules de code<br>
    
<em>  
To make nice html reports with markdown: <a href="https://dillinger.io/" title="dillinger.io">html visualization tool 1</a> or <a href="https://stackedit.io/app#" title="stackedit.io">html visualization tool 2</a>, <a href="https://www.tablesgenerator.com/markdown_tables" title="tablesgenerator.com">to draw nice tables</a>, and the <a href="https://medium.com/analytics-vidhya/the-ultimate-markdown-guide-for-jupyter-notebook-d5e5abf728fd" title="Ultimate guide">Ultimate guide</a>. <br>
Further reading on JupyterLab notebooks: <a href="https://jupyterlab.readthedocs.io/en/latest/user/notebook.html" title="Jupyter Lab">Jupyter Lab documentation</a>.<br>
Here we are using JupyterLab interface implemented as part of the <a href="https://plasmabio.org/" title="plasmabio.org">Plasmabio</a> project led by Sandrine Caburet, Pierre Poulain and Claire Vandiedonck.
</em>    
</div>

Les informations de configuration de ce notebook et de l'environnement jupyterlab associé sont disponibles à ce lien: https://github.com/CVandiedonck/T1D_GWAS_long_version

*[last version: 24/02/2026 by Claire Vandiedonck]*

---